In [8]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash
JupyterDash.infer_jupyter_proxy_config()

# Dash and visualization imports
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import dash_leaflet as dl
import plotly.express as px

# Data / utility imports
import pandas as pd
import base64

# Import CRUD module
from CRUD_Python_Module import AnimalShelter

###########################
# Database Connection
###########################
username = "aacuser"
password = "SNHU1234"

db = AnimalShelter(username, password)

df = pd.DataFrame.from_records(db.read({}))

###########################
# App Initialization
###########################
app = JupyterDash(__name__)

###########################
# Load Logo
###########################
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

###########################
# Dashboard Layout
###########################
app.layout = html.Div([

    # Logo and Title
    html.Center([
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            width='200px'
        ),
        html.H2("Grazioso Salvare Rescue Dashboard"),
        html.H4("Seunghwan Lee – CS 340 Project Two")
    ]),

    html.Hr(),

    # Filter Options
    html.Div([
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain / Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster / Individual Tracking', 'value': 'disaster'},
                {'label': 'Reset', 'value': 'reset'}
            ],
            value='reset',
            inline=True
        )
    ]),

    html.Hr(),

    # Data Table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[0],
        style_table={'overflowX': 'auto'}
    ),

    html.Br(),
    html.Hr(),

    # Charts (Side-by-side)
    html.Div(style={'display': 'flex'}, children=[
        html.Div(id='graph-id', style={'width': '50%'}),
        html.Div(id='map-id', style={'width': '50%'})
    ])
])

#############################################
# Filtering Callback (Controller)
#############################################
@app.callback(
    Output('datatable-id', 'data'),
    Input('filter-type', 'value')
)
def update_table(filter_type):

    if filter_type == 'water':
        query = {
            "breed": {"$in": [
                "Labrador Retriever Mix",
                "Chesapeake Bay Retriever",
                "Newfoundland"
            ]},
            "age_upon_outcome_in_weeks": {"$lte": 104}
        }

    elif filter_type == 'mountain':
        query = {
            "breed": {"$in": [
                "German Shepherd",
                "Alaskan Malamute",
                "Old English Sheepdog"
            ]},
            "age_upon_outcome_in_weeks": {"$lte": 104}
        }

    elif filter_type == 'disaster':
        query = {
            "breed": {"$in": [
                "Doberman Pinscher",
                "Rottweiler",
                "Bloodhound"
            ]},
            "age_upon_outcome_in_weeks": {"$lte": 104}
        }

    else:
        query = {}

    df_filtered = pd.DataFrame.from_records(db.read(query))
    return df_filtered.to_dict('records')

#############################################
# Breed Distribution Chart
#############################################
@app.callback(
    Output('graph-id', "children"),
    Input('datatable-id', "derived_virtual_data")
)
def update_graph(viewData):

    if viewData is None or len(viewData) == 0:
        return

    dff = pd.DataFrame(viewData)

    fig = px.histogram(
        dff,
        x="breed",
        title="Breed Distribution"
    )

    return dcc.Graph(figure=fig)

#############################################
# Geolocation Map
#############################################
@app.callback(
    Output('map-id', "children"),
    [
        Input('datatable-id', "derived_virtual_data"),
        Input('datatable-id', "derived_virtual_selected_rows")
    ]
)
def update_map(viewData, selected_rows):

    if viewData is None or selected_rows is None:
        return

    dff = pd.DataFrame(viewData)
    row = selected_rows[0]

    return dl.Map(
        style={'width': '100%', 'height': '500px'},
        center=[30.75, -97.48],
        zoom=10,
        children=[
            dl.TileLayer(),
            dl.Marker(
                position=[dff.iloc[row]["location_lat"], dff.iloc[row]["location_long"]],
                children=[
                    dl.Tooltip(dff.iloc[row]["breed"]),
                    dl.Popup([
                        html.H4("Animal Name"),
                        html.P(dff.iloc[row]["name"])
                    ])
                ]
            )
        ]
    )

###########################
# Run App
###########################
app.run_server()

Dash app running on https://omegalibra-admiralcinema-3000.codio.io/proxy/8050/
